In [140]:
import numpy as np
import gurobipy as gp
from gurobipy import GRB
# from TakeTwoSetup import *

n = 4
m = 4
holdingCost = [1000, 1200, 800, 1300]
durations = [50, 30, 60, 80]
S = np.matrix([
  [0, 100, 10, 10],
  [40, 0, 60, 50],
  [40, 90, 0, 30],
  [20, 60, 20, 0]
])
W = 200

# --- Finding D and H --- #
# D: max number of days
# H: max number of scenes in a day
def find_D(n, S, durations, W):
  D = (2*n*(max(durations)+np.matrix.max(S)))/(W+np.matrix.max(S))
  if np.ceil(D)%2 == 0:
    D = int(np.floor(D))
  else:
    D=int(np.ceil(D))
  return D

D = find_D(n, S, durations, W)

def find_H(n, S, durations, W):
  increasingDurations = np.sort(durations)
  increasingChangeover = np.array(np.sort(np.matrix.flatten(S)))[0]

  vals = []
  hs = []
  for h in range(2, n+1):
    total = np.sum(increasingChangeover[:h]) + np.sum(increasingDurations[:h])
    if total <= W:
      # to make sure combination is within admissible set
      vals.append(total)
      hs.append(h)
  H = hs[np.argmax(vals)]
  return H

H = find_H(n, S, durations, W)

D, H

(5, 3)

In [141]:
from itertools import permutations

def finding_P(n, S, durations, W, H):
    valid_perms = []
    
    for length in range(H):
        for perm in permutations(range(n), length+1):
            total = 0
            
            for scene in perm:
                total += durations[scene]
            
            for i in range(len(perm)-1):
                total += S[perm[i], perm[i+1]]
            
            if total <= W:
                valid_perms.append(perm)
    
    return valid_perms

# unique subsets from valid permutations
def get_unique_subsets_from_perms(valid_perms):
    subsets = set()
    for perm in valid_perms:
        subsets.add(frozenset(perm))
    return list(subsets)

P_sets = finding_P(n, S, durations, W, H)
P_sets = get_unique_subsets_from_perms(P_sets)
P = [set(subset) for subset in P_sets]
k = len(P)
k, P

(11,
 [{2},
  {2, 3},
  {1, 2},
  {0, 3},
  {0, 1, 2},
  {0, 1},
  {0, 2},
  {3},
  {1},
  {1, 3},
  {0}])

In [142]:
# --- Scene j in P_k (matrix A) --- #
A = np.zeros((k, n), dtype=int)

for j, subset in enumerate(P_sets):
    for i in range(n):
        if i in subset:
            A[j, i] = 1

A

array([[0, 0, 1, 0],
       [0, 0, 1, 1],
       [0, 1, 1, 0],
       [1, 0, 0, 1],
       [1, 1, 1, 0],
       [1, 1, 0, 0],
       [1, 0, 1, 0],
       [0, 0, 0, 1],
       [0, 1, 0, 0],
       [0, 1, 0, 1],
       [1, 0, 0, 0]])

In [143]:
# --- ILP_pat Model Setup --- #
ILP_pat = gp.Model("ILP_pat", env=env)

# decision variables
x_pat = ILP_pat.addVars(k, D, vtype=GRB.BINARY, name="x_pat")
y_pat = ILP_pat.addVars(m, D, D, vtype=GRB.BINARY, name="y_pat")

# objective function
objFunc_pat = ILP_pat.setObjective(
    gp.quicksum(
        holdingCost[i] *
        gp.quicksum(
              (d2-d1+1)*y_pat[i, d1, d2]
              for d1 in range(D)
              for d2 in range(d1, D)
        )
        for i in range(m)
    ), GRB.MINIMIZE
)

# --- constraints (lord help us) --- #
c21 = ILP_pat.addConstr(
    gp.quicksum(
        x_pat[k, 1] for k in range(len(P))
    ) == 1, name="c21"
) # Constraint (21)

c22 = ILP_pat.addConstrs((
    gp.quicksum(
        x_pat[k, d-1]
        for k in range(len(P))
        ) >=
    gp.quicksum(
        x_pat[k, d]
        for k in range(len(P))
        )
    for d in range(2, D)),
    name="c22"
) # Constraint (22)

c23 = ILP_pat.addConstrs((
    gp.quicksum(
        A[k, j] * x_pat[k, d]
        for k in range(len(P))
        for d in range(D)
    ) == 1
    for j in range(n)),
    name="c23"
) # Constraint (23)

c24 = ILP_pat.addConstrs((
    gp.quicksum(
        y_pat[i, d1, d2]
        for d1 in range(D)
        for d2 in range(d1, D)
    ) == 1
    for i in range(m)),
    name="c24"
) # Constraint (24)

c25 = ILP_pat.addConstrs((
    gp.quicksum(
        T[i, j] * A[k, j] * x_pat[k, d]
        for k in range(len(P))
        for j in range(n)
    ) <=
    gp.quicksum(
        y_pat[i, d1, d2]
        for d2 in range(d, D)
        for d1 in range(d)
    )
    for i in range(m) for d in range(D)),
    name="c25"
) # Constraint (25)

# --- Constraints (26-27) --- #
# These constraints are embedded in the init of the variables

In [144]:
# --- optimize that shit --- #
ILP_pat.setParam('TimeLimit', 360)
ILP_pat.optimize()
sol = ILP_pat.getJSONSolution()

Set parameter TimeLimit to value 360
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: AMD Ryzen 7 4700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  360

Academic license 2773076 - for non-commercial use only - registered to km___@uga.edu
Optimize a model with 32 rows, 155 columns and 462 nonzeros (Min)
Model fingerprint: 0xc9789949
Model has 60 linear objective coefficients
Variable types: 0 continuous, 155 integer (155 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [8e+02, 7e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 17200.000000
Presolve removed 6 rows and 105 columns
Presolve time: 0.01s
Presolved: 26 rows, 50 columns, 169 nonzeros
Variable types: 0 continuous, 50 integer (50 binary)

Root relaxation: objective 1.30

In [145]:
all_vars = ILP_pat.getVars()
values = ILP_pat.getAttr("X", all_vars)
names = ILP_pat.getAttr("VarName", all_vars)

for name, val in zip(names, values):
    if val == 1:
        print(f"{name} = {val}")

x_pat[1,3] = 1.0
x_pat[8,2] = 1.0
x_pat[10,1] = 1.0
y_pat[0,0,3] = 1.0
y_pat[1,0,3] = 1.0
y_pat[2,1,2] = 1.0
y_pat[3,1,2] = 1.0


In [148]:
P[1], P[8], P[10]

({2, 3}, {1}, {0})

In [146]:
all_vars = ILP_pat.getVars()
values = ILP_pat.getAttr("X", all_vars)
names = ILP_pat.getAttr("VarName", all_vars)

for name, val in zip(names, values):
    if val == 1:
        print(f"{name} = {val}")

x_pat[1,3] = 1.0
x_pat[8,2] = 1.0
x_pat[10,1] = 1.0
y_pat[0,0,3] = 1.0
y_pat[1,0,3] = 1.0
y_pat[2,1,2] = 1.0
y_pat[3,1,2] = 1.0


In [147]:
with open("sol.json", "w") as f:
  f.write(sol)